In [1]:
import sys
import os

# Agregar el directorio padre al sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime


import cashia_core.data_channel.promok_datapipe_func as pkdp
import cashia_core.common_tools.pm_mxgeographic as pm_mxgeo

import mlp.promok_models_config as pkconfig
from mlp.dataframe_tracker import *


%matplotlib inline

In [2]:
import sklearn as sk

print("numpy:",np.__version__)
print("pandas:",pd.__version__)
print("seborn:",sns.__version__)
print("sklearn:",sk.__version__)
#print(":",.__version__)

numpy: 1.26.0
pandas: 1.5.3
seborn: 0.13.2
sklearn: 1.2.2


# 1 Procesar los datos

## 1.1 Lectura de los datos

### 1.1.1 Lectura de los datos de configuración

In [3]:
print('Configuration file: ', pkconfig.configuration_file_name)
print('Configuration : ', pkconfig.conf_column)
all_features = pd.read_excel("./config/" + pkconfig.configuration_file_name, index_col=0)

Configuration file:  CashIA_ConfFile.xlsx
Configuration :  NV_CC_CS


c:\Users\juan_\.conda\envs\cashia_env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [4]:
features = all_features[all_features[pkconfig.conf_column] == 'Sí']
features

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
9,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Monto,Predictor,1,900.0,17000.0,Cuantitativo,Entero,1,Crédito,NaN,x
11,Sí,Sí,Sí,No,Sí,Sí,Sí,Sí,Edad Solicitud,Predictor,1,18.0,100.0,Cuantitativo,Entero,0,Datos personales,Edad al hacer la solicitud,NaN
15,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Genero,Predictor,0,NaN,NaN,Categórico,Texto,0,Datos personales,NaN,NaN
18,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Dependientes,Predictor,1,0.0,10.0,Cuantitativo,Entero,0,Datos personales,NaN,x
34,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Ingresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de ingresos,NaN,x
35,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Egresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de egresos,NaN,x
51,Sí- Off,Sí- Off,Sí,Sí,Sí- Off,Sí- Off,Sí,Sí,Score CC,Predictor,1,0.0,820.0,Cuantitativo,Entero,0,Ciculo de Crédito,NaN,x
82,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Mes inicial,Predictor,1,1.0,12.0,Cuantitativo,Entero,0,Data Agregation,NaN,NaN
83,No,No,Sí,Sí,No,No,Sí,Sí,Odds,Predictor,1,0.0,9.0,Cuantitativo,Real,0,Data Agregation,NaN,x


In [5]:
features_to_use = set(features['Característica'].values)
features_to_use

{'Deficit',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Genero',
 'Incobrable',
 'IngresoPercapita',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Odds',
 'Score CC'}

In [6]:
agregated_feautures = set(features[features['Sección'] == 'Data Agregation']['Característica'].values)
agregated_feautures

{'Deficit', 'IngresoPercapita', 'Mes inicial', 'Odds'}

### 1.1.2 Lectura de los datos de entrenamiento

In [7]:
print('Source file: ', pkconfig.source_file_name)
#data = pd.read_excel(pkconfig.source_file_name)
data = pd.read_csv("./data/" + pkconfig.source_file_name, index_col=0)

data_tracker = DataFrameEvolutionTracker(data)
data_tracker.to_dataframe()

if pkconfig.conf_column == "NV_Agt" or pkconfig.conf_column == "RNV_Agt":
    print("Agent score")
    data = data[data['Score Agt'] != 0]
else:
    print("Using CC score")

data_tracker.register_step(f"First filter for {pkconfig.conf_column}", data)
data_tracker.to_dataframe()

data.head()

Source file:  Data Set Cashia Entrenamiento.csv
Using CC score


C:\Users\juan_\AppData\Local\Temp\ipykernel_35872\725869802.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("./data/" + pkconfig.source_file_name, index_col=0)


,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0.000000,13,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.00000
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,0.000000,23,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.00000
66494,k66494000064945,6494,30/12/2022,05/05/2023,11000,24,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20458,0.0,2005,0.05882,561.0,17.0,0.00000
66857,k66857000065890,6589,03/01/2023,04/05/2023,14000,45,Femenino,0,72014,72014.0,...,1.063799,18,0,0.20455,0.0,2005,0.05882,561.0,17.0,0.05882
72961,k7296110400351909,35190,21/02/2023,03/08/2023,5000,29,Femenino,0,90790,90790.0,...,1.198630,24,1,0.20455,0.0,2107,0.05882,568.0,17.0,0.00000


# Filtros especiales

## Borramos los datos de créditos no aprobados

In [8]:
# Quitamos esta línea en diciembre 2025 ya que en teoría sólo me pasan los créditos aprobados
# data = data[data['Aprobado'] == 1]
# data.shape

## De acuerdo a la columna de configuración selecionamos los datos

In [9]:
# Caso 1: Renovaciones
if pkconfig.conf_column == "NV_Agt":
    data = data[data['Categoria'] == "NV"]
    
elif pkconfig.conf_column == "RNV_Agt":
    data = data[(data['Categoria'] == "RNV")]
    
elif pkconfig.conf_column == "NV_CC":
    data = data[(data['Categoria'] == "NV")]
    
elif pkconfig.conf_column == "RNV_CC":
    data = data[(data['Categoria'] == "RNV")]

elif pkconfig.conf_column == "NV_Agt_CS":
    data = data[data['Categoria'] == "NV"]
    
elif pkconfig.conf_column == "RNV_Agt_CS":
    data = data[(data['Categoria'] == "RNV")]
    
elif pkconfig.conf_column == "NV_CC_CS":
    data = data[(data['Categoria'] == "NV")]
    
elif pkconfig.conf_column == "RNV_CC_CS":
    data = data[(data['Categoria'] == "RNV")]
    
else:
    print("Invalid option")
    exit()

data_tracker.register_step(f"Second filter for {pkconfig.conf_column}", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


## 1.2 Explorar los datos

In [10]:
data_features = set(data.columns)
data_features

{'CP',
 'CP Aval',
 'Categoria',
 'Ciclo',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Fecha Inicial',
 'Frecuencia_Ant',
 'Genero',
 'Incobrable',
 'Ingresos',
 'Monto',
 'Monto_Ant',
 'Plazo',
 'Score Agt',
 'Score CC',
 'Score_Ant',
 'Unidad',
 'fecha_finalizada',
 'id_agente',
 'id_cliente',
 'id_credito_ant',
 'id_fecha_finalizada_actual',
 'id_fecha_finalizada_ant',
 'id_fecha_impresion_actual',
 'id_fecha_impresion_ant',
 'pagado_cred_ant',
 'pagos_realizados_actual',
 'referencia',
 'score',
 'score_agente',
 'sem_inicio',
 'semanas',
 'tarifa_ant',
 'vida_actual',
 'vida_ant'}

In [11]:
features_to_drop = list(data_features - features_to_use)
# Removemos de la lista de elementos a borrar 'Fecha Inicial ya que la necesitaremos para los intervalos de aprendizaje'
features_to_drop.remove('Fecha Inicial')
features_to_drop

['Unidad',
 'fecha_finalizada',
 'semanas',
 'sem_inicio',
 'Score Agt',
 'Frecuencia_Ant',
 'id_fecha_finalizada_ant',
 'Monto_Ant',
 'vida_actual',
 'id_cliente',
 'referencia',
 'id_credito_ant',
 'score_agente',
 'Plazo',
 'Ciclo',
 'id_fecha_finalizada_actual',
 'CP Aval',
 'CP',
 'id_agente',
 'id_fecha_impresion_actual',
 'Score_Ant',
 'Categoria',
 'score',
 'tarifa_ant',
 'id_fecha_impresion_ant',
 'pagado_cred_ant',
 'vida_ant',
 'pagos_realizados_actual']

In [12]:
features

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
9,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Monto,Predictor,1,900.0,17000.0,Cuantitativo,Entero,1,Crédito,NaN,x
11,Sí,Sí,Sí,No,Sí,Sí,Sí,Sí,Edad Solicitud,Predictor,1,18.0,100.0,Cuantitativo,Entero,0,Datos personales,Edad al hacer la solicitud,NaN
15,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Genero,Predictor,0,NaN,NaN,Categórico,Texto,0,Datos personales,NaN,NaN
18,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Dependientes,Predictor,1,0.0,10.0,Cuantitativo,Entero,0,Datos personales,NaN,x
34,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Ingresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de ingresos,NaN,x
35,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Egresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de egresos,NaN,x
51,Sí- Off,Sí- Off,Sí,Sí,Sí- Off,Sí- Off,Sí,Sí,Score CC,Predictor,1,0.0,820.0,Cuantitativo,Entero,0,Ciculo de Crédito,NaN,x
82,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Mes inicial,Predictor,1,1.0,12.0,Cuantitativo,Entero,0,Data Agregation,NaN,NaN
83,No,No,Sí,Sí,No,No,Sí,Sí,Odds,Predictor,1,0.0,9.0,Cuantitativo,Real,0,Data Agregation,NaN,x


In [13]:
int_features_list = features[(features['Sub tipo'] == "Entero") & (features['Sección'] != "Data Agregation")]
int_features_list = int_features_list['Característica'].unique()
int_features_list = list(int_features_list)

In [14]:
int_features_list

['Monto',
 'Edad Solicitud',
 'Dependientes',
 'Ingresos',
 'Egresos',
 'Score CC',
 'Incobrable']

In [15]:
all_features

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
1,No,No,No,No,No,No,No,No,Id Solicitud,Predictor,1,0.0,999999.0,Cuantitativo entero,Entero,0,NaN,NaN,NaN
2,Sí- Off,No,Sí- Off,Sí- Off,Sí- Off,No,Sí- Off,Sí- Off,Plazo,Predictor,1,9.0,27.0,Categórico,Entero,0,Crédito,NaN,NaN
3,No,No,Sí- Off,Sí- Off,No,No,Sí- Off,Sí- Off,Categoria,Predictor,0,NaN,NaN,Categórico,Texto,0,Crédito,NaN,NaN
4,Sí,Sí,Sí,Sí,No,No,No,No,Unidad,Predictor,0,NaN,NaN,Categórico,Texto,0,Crédito,NaN,x
5,No,No,No,No,No,No,No,No,Zona,Predictor,0,NaN,NaN,Categórico,Texto,0,Crédito,"Mapearlo a ""unidad"" ya que son muchos valores",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,Sí,Sí,Sí,Sí,No,No,No,No,CP_Longitud,Predictor,1,-180.0,180.0,Cuantitativo,Real,0,Data Agregation,NaN,x
90,Sí- Off,No,Sí,Sí,Sí- Off,No,No,No,CP_Aval_Latitud,Predictor,1,-90.0,90.0,Cuantitativo,Real,0,Data Agregation,NaN,x
91,Sí- Off,No,Sí,Sí,Sí- Off,No,No,No,CP_Aval_Longitud,Predictor,1,-180.0,180.0,Cuantitativo,Real,0,Data Agregation,NaN,x


In [16]:
if ('CP' not in features_to_use) and (('CP_Latitud' in features_to_use)| ('CP_Longitud' in features_to_use)):
    int_features_list.append('CP')
    
if ('CP Aval' not in features_to_use) and (('CP_Aval_Latitud' in features_to_use) | ('CP_Aval_Longitud' in features_to_use)):
    int_features_list.append('CP Aval')                                        

if ('Score CC' not in features_to_use) and 'Odds' in features_to_use:
    int_features_list.append('Score CC')

In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48024 entries, 71677 to 124200
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   referencia                  48024 non-null  object 
 1   id_cliente                  48024 non-null  int64  
 2   Fecha Inicial               48024 non-null  object 
 3   fecha_finalizada            32759 non-null  object 
 4   Monto                       48024 non-null  int64  
 5   Edad Solicitud              48024 non-null  int64  
 6   Genero                      48024 non-null  object 
 7   Dependientes                48024 non-null  int64  
 8   CP                          48024 non-null  int64  
 9   CP Aval                     47435 non-null  object 
 10  Ingresos                    48003 non-null  float64
 11  Egresos                     48024 non-null  float64
 12  Score CC                    48024 non-null  object 
 13  Plazo                     

In [18]:
data.isnull().sum()

referencia                        0
id_cliente                        0
Fecha Inicial                     0
fecha_finalizada              15265
Monto                             0
Edad Solicitud                    0
Genero                            0
Dependientes                      0
CP                                0
CP Aval                         589
Ingresos                         21
Egresos                           0
Score CC                          0
Plazo                             0
Categoria                         0
Unidad                            0
Ciclo                             0
id_credito_ant                48011
Monto_Ant                     48011
tarifa_ant                    48011
pagado_cred_ant               48011
id_fecha_impresion_ant        48011
id_fecha_finalizada_ant       48011
id_fecha_impresion_actual         0
id_fecha_finalizada_actual        0
pagos_realizados_actual           0
vida_ant                      48011
Frecuencia_Ant              

## 1.3 Limpiar los datos

In [19]:
quantitative_features = set(features[(features['Tipo de dato'] == 'Cuantitativo') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
quantitative_features

{'Deficit',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'IngresoPercapita',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Odds',
 'Score CC'}

In [20]:
categorical_features = set(features[(features['Tipo de dato'] == 'Categórico') & (features['Tipo de variable'] == 'Predictor')]['Característica'].values)
categorical_features

{'Genero'}

### 1.3.1 Limpiar ingresos de conyuge para los que tienen NA poner "Desconocido"

In [21]:
if 'Ingreso Conyuge' in features_to_use: 
    pkdp.clean_ingreso_conyuge(data)
data_tracker.register_step(f"Cleaning datos conyuge", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


### 1.3.1b

In [22]:
#pk.clean_incobrable(data)

### 1.3.2 Poner los egresos igual a sus ingresos a aquellos que tienen vacío en egresos

In [23]:
pkdp.clean_egresos(data)
data_tracker.register_step(f"Cleaning egresos", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [24]:
data.isnull().sum()

referencia                        0
id_cliente                        0
Fecha Inicial                     0
fecha_finalizada              15265
Monto                             0
Edad Solicitud                    0
Genero                            0
Dependientes                      0
CP                                0
CP Aval                         589
Ingresos                         21
Egresos                           0
Score CC                          0
Plazo                             0
Categoria                         0
Unidad                            0
Ciclo                             0
id_credito_ant                48011
Monto_Ant                     48011
tarifa_ant                    48011
pagado_cred_ant               48011
id_fecha_impresion_ant        48011
id_fecha_finalizada_ant       48011
id_fecha_impresion_actual         0
id_fecha_finalizada_actual        0
pagos_realizados_actual           0
vida_ant                      48011
Frecuencia_Ant              

In [25]:
raw_data = features_to_use-agregated_feautures
raw_data

{'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'Genero',
 'Incobrable',
 'Ingresos',
 'Monto',
 'Score CC'}

In [26]:
data[list(raw_data)].isnull().sum()

Monto              0
Egresos            0
Ingresos          21
Genero             0
Edad Solicitud     0
Dependientes       0
Score CC           0
Incobrable         0
dtype: int64

### 1.3.3. Llenamos con "Unknown" los valores de columnas categóricas vacias

In [27]:
#data[list(categorical_features-agregated_feautures)] = data[list(categorical_features-agregated_feautures)].fillna('Unknown')

### 1.3.4 Borrar renglones con datos nulos en las columnas que se usarán en el modelo

In [28]:
data = data.dropna(subset=list(features_to_use-agregated_feautures))
data_tracker.register_step(f"Cleaning for columns with null", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,48003,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [29]:
data.isnull().sum()

referencia                        0
id_cliente                        0
Fecha Inicial                     0
fecha_finalizada              15255
Monto                             0
Edad Solicitud                    0
Genero                            0
Dependientes                      0
CP                                0
CP Aval                         589
Ingresos                          0
Egresos                           0
Score CC                          0
Plazo                             0
Categoria                         0
Unidad                            0
Ciclo                             0
id_credito_ant                47990
Monto_Ant                     47990
tarifa_ant                    47990
pagado_cred_ant               47990
id_fecha_impresion_ant        47990
id_fecha_finalizada_ant       47990
id_fecha_impresion_actual         0
id_fecha_finalizada_actual        0
pagos_realizados_actual           0
vida_ant                      47990
Frecuencia_Ant              

### 1.3.5 Convertir a entero las columnas que deberían ser entero y no lo son

In [30]:
# 1) Convertir a numérico con NaN donde haya "NC" o vacío
data[int_features_list] = data[int_features_list].apply(
    pd.to_numeric, errors='coerce', downcast='integer'
)
data[int_features_list].isnull().sum()

Monto                 0
Edad Solicitud        0
Dependientes          0
Ingresos              0
Egresos               0
Score CC          21912
Incobrable            0
dtype: int64

In [31]:
# 2) Eliminar filas con NaN en esas columnas
data = data.dropna(subset=int_features_list)

# 3) Forzar a int (si quieres asegurarte que sean int "normales")
data[int_features_list] = data[int_features_list].astype(int)

In [32]:
data_tracker.register_step(f"Cleaning int features with dropna", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,48003,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,26091,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


### 1.3.5 Convertir a float (Real) las columnas que deberían ser flot y no lo son

In [33]:
float_features_list = features[(features['Sub tipo'] == "Real") & (features['Sección'] != "Data Agregation")]['Característica'].unique()
float_features_list = list(float_features_list)

In [34]:
# if ('Score CC' not in features_to_use) and ('Odds' in features_to_use):
#     float_features_list.append('Score CC')    
# display(float_features_list)

In [35]:
# Convertir todo a floats (con NaN donde haya errores)
data[float_features_list] = data[float_features_list].apply(
    lambda col: pd.to_numeric(col, errors='coerce')
)

# Asegurar float64
data[float_features_list] = data[float_features_list].astype(float)

# Eliminar filas con problemas
data = data.dropna(subset=float_features_list)

print(data[float_features_list].info())
data_tracker.register_step(f"Cleaning na floats", data)
data_tracker.to_dataframe()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 26091 entries, 71677 to 124200
Empty DataFrame
None


,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,48003,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,26091,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
7,Cleaning na floats,26091,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


### 1.3.6 Identificar características cuantitativas y cualitativas

In [36]:
for feature in quantitative_features : 
    if feature not in agregated_feautures:
        pkdp.categorical_print_values(feature, data)
        print("-----------------------")

Monto
3000     13086
5000      6581
4000      1948
3500       675
10000      593
6000       505
7000       414
4500       386
8000       349
2000       303
15000      228
17000      147
12000      103
9000        91
5500        85
11000       67
2500        66
6500        54
7500        50
13000       29
8500        25
10500       20
9500        19
4800        18
3800        18
3900        18
3300        18
4900        15
3700        15
4600        14
3600        14
3400        14
3200        13
12500       12
3100        12
11500       11
4400        10
4100        10
14000        8
4300         8
13500        7
16000        6
4700         6
4200         5
2200         4
15500        3
2300         2
2700         1
12300        1
5100         1
14500        1
2100         1
2600         1
Name: Monto, dtype: int64
-----------------------
Egresos
5000     2673
0        2640
4000     2295
6000     1941
3000     1811
         ... 
8750        1
2870        1
56000       1
2698        1
2

In [37]:
data.head(5)

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0.0,13,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.00000
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,0.0,23,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.00000
69758,k6975899900395917,39591,25/01/2023,12/05/2023,3000,18,Femenino,0,62748,62748.0,...,0.0,16,0,0.20456,0.0,2206,0.17940,564.0,17.0,0.00000
72809,k7280999900413583,41358,15/02/2023,05/05/2023,3000,62,Femenino,0,62748,62748.0,...,0.0,12,0,0.20456,0.0,2206,0.16130,567.0,17.0,0.17035
66301,k6630110001200377482,37748,02/01/2023,30/03/2023,3000,35,Femenino,0,94347,94347.0,...,0.0,13,0,0.20456,0.0,2207,0.17395,561.0,17.0,0.00000


In [38]:
for feature in categorical_features :
    if feature not in agregated_feautures:
        pkdp.categorical_print_values(feature, data)
        print("-----------------------")

Genero
Femenino     18753
Masculino     7338
Name: Genero, dtype: int64
-----------------------


### 1.3.7 Limpiar datos numéricos

In [39]:
numeric_features = features[(features['Numérico'] == 1) & (features['Sección'] != 'Data Agregation')]  
display(numeric_features)

,NV_Agt,RNV_Agt,NV_CC,RNV_CC,NV_Agt_CS,RNV_Agt_CS,NV_CC_CS,RNV_CC_CS,Característica,Tipo de variable,Numérico,Min,Max,Tipo de dato,Sub tipo,Monetario,Sección,Observación,Unnamed: 19
ID,,,,,,,,,,,,,,,,,,,
9,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Monto,Predictor,1,900.0,17000.0,Cuantitativo,Entero,1,Crédito,NaN,x
11,Sí,Sí,Sí,No,Sí,Sí,Sí,Sí,Edad Solicitud,Predictor,1,18.0,100.0,Cuantitativo,Entero,0,Datos personales,Edad al hacer la solicitud,NaN
18,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Dependientes,Predictor,1,0.0,10.0,Cuantitativo,Entero,0,Datos personales,NaN,x
34,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Ingresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de ingresos,NaN,x
35,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Egresos,Predictor,1,0.0,50000.0,Cuantitativo,Entero,1,Estudio de egresos,NaN,x
51,Sí- Off,Sí- Off,Sí,Sí,Sí- Off,Sí- Off,Sí,Sí,Score CC,Predictor,1,0.0,820.0,Cuantitativo,Entero,0,Ciculo de Crédito,NaN,x
93,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Incobrable,Target,1,0.0,1.0,Categórico,Entero,0,NaN,NaN,NaN


In [40]:
pkdp.print_sizes(data)
for index, row in numeric_features.iterrows():
    print(f"Column {row['Característica']} max value found: {data[row['Característica']].max()}")
    pkdp.clean_numeric_data(data, row['Característica'], row['Min'], row['Max'])
    print("=================================================")
pkdp.print_sizes(data)

Rows: 26091
Columns: 37
Max index: 264422
Column Monto max value found: 17000
..=================================================
Column Edad Solicitud max value found: 70
..=================================================
Column Dependientes max value found: 10
..=================================================
Column Ingresos max value found: 185500
..Total of incorrect registers: 43
For example: minvalue:0.0 maxvalue:50000.0 found 68000
Column Egresos max value found: 600000
..Total of incorrect registers: 28
For example: minvalue:0.0 maxvalue:50000.0 found 90000
Column Score CC max value found: 769
..=================================================
Column Incobrable max value found: 1
..=================================================
Rows: 26020
Columns: 37
Max index: 264422


In [41]:
data_tracker.register_step(f"Cleaning numeric data min max", data)
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,48003,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,26091,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
7,Cleaning na floats,26091,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
8,Cleaning numeric data min max,26020,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [42]:
data.head()

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Frecuencia_Ant,vida_actual,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0.0,13,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.00000
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,0.0,23,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.00000
69758,k6975899900395917,39591,25/01/2023,12/05/2023,3000,18,Femenino,0,62748,62748.0,...,0.0,16,0,0.20456,0.0,2206,0.17940,564.0,17.0,0.00000
72809,k7280999900413583,41358,15/02/2023,05/05/2023,3000,62,Femenino,0,62748,62748.0,...,0.0,12,0,0.20456,0.0,2206,0.16130,567.0,17.0,0.17035
66301,k6630110001200377482,37748,02/01/2023,30/03/2023,3000,35,Femenino,0,94347,94347.0,...,0.0,13,0,0.20456,0.0,2207,0.17395,561.0,17.0,0.00000


# 2 Data agregation

## 2.1 Agregar $Deficit = Ingresos - Egresos$

In [43]:
if 'Deficit' in quantitative_features:
    data['Deficit'] = data['Ingresos']-data['Egresos']
#quantitative_features.add('Deficit')

## 2.2 Agregar ingreso per capita

In [44]:
if 'IngresoPercapita' in  quantitative_features:
    data['IngresoPercapita'] = data['Ingresos']/(data['Dependientes']+1)
#quantitative_features.add('IngresoPercapita')

## 2.2 Agregar la zona a partir del Codigo Postal

In [45]:
#if 'TipoZona' in categorical_features:
#    CP_to_Zone = pd.read_excel("./CP_To_Zone.xlsx", index_col=0)
#    CP_Zone_set = set(CP_to_Zone.index)
#    data['TipoZona'] = data.apply(lambda x: CP_to_Zone.loc[x['CP']].Zona if x['CP'] in CP_Zone_set 
#                                  else "Unknown", axis=1)
#    data.head(5)

In [46]:
cp_manager = pm_mxgeo.MX_CP_manager('./mlp/inputs/MX_CP_CoordZone.csv')

In [47]:
data.columns

Index(['referencia', 'id_cliente', 'Fecha Inicial', 'fecha_finalizada',
       'Monto', 'Edad Solicitud', 'Genero', 'Dependientes', 'CP', 'CP Aval',
       'Ingresos', 'Egresos', 'Score CC', 'Plazo', 'Categoria', 'Unidad',
       'Ciclo', 'id_credito_ant', 'Monto_Ant', 'tarifa_ant', 'pagado_cred_ant',
       'id_fecha_impresion_ant', 'id_fecha_finalizada_ant',
       'id_fecha_impresion_actual', 'id_fecha_finalizada_actual',
       'pagos_realizados_actual', 'vida_ant', 'Frecuencia_Ant', 'vida_actual',
       'Incobrable', 'score', 'Score_Ant', 'id_agente', 'score_agente',
       'sem_inicio', 'semanas', 'Score Agt', 'Deficit', 'IngresoPercapita'],
      dtype='object')

In [48]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt,Deficit,IngresoPercapita
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.000000,-4000,0.0
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.000000,3000,6000.0
69758,k6975899900395917,39591,25/01/2023,12/05/2023,3000,18,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.17940,564.0,17.0,0.000000,3200,7200.0
72809,k7280999900413583,41358,15/02/2023,05/05/2023,3000,62,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.16130,567.0,17.0,0.170350,-3000,0.0
66301,k6630110001200377482,37748,02/01/2023,30/03/2023,3000,35,Femenino,0,94347,94347.0,...,0,0.20456,0.0,2207,0.17395,561.0,17.0,0.000000,11000,15600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86913,k8691300900493934,49393,31/05/2023,13/02/2025,5000,40,Femenino,0,72770,72770.0,...,1,0.05814,0.0,P1594A,0.04904,657.0,16.0,1.760470,4000,8000.0
94056,k9405653593,53593,19/07/2023,15/12/2024,5000,29,Femenino,0,72840,72803.0,...,1,0.10624,0.0,P1594A,1.40224,661.0,3.0,1.515980,6000,6000.0
101852,k10185257870,57870,09/09/2023,28/11/2024,3000,19,Femenino,0,72803,72803.0,...,1,0.19025,0.0,P1594A,0.27397,657.0,5.0,1.147565,6500,6500.0


In [49]:
if 'TipoZona' in categorical_features:
    data['index'] = data.index
    print(data.head(5))
    data = data.merge(cp_manager.cp_base[['CP', 'TipoZona']], left_on='CP', right_on='CP', how='left').fillna("Fake")
    data = data.set_index(['index'])

In [50]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt,Deficit,IngresoPercapita
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.000000,-4000,0.0
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.000000,3000,6000.0
69758,k6975899900395917,39591,25/01/2023,12/05/2023,3000,18,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.17940,564.0,17.0,0.000000,3200,7200.0
72809,k7280999900413583,41358,15/02/2023,05/05/2023,3000,62,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.16130,567.0,17.0,0.170350,-3000,0.0
66301,k6630110001200377482,37748,02/01/2023,30/03/2023,3000,35,Femenino,0,94347,94347.0,...,0,0.20456,0.0,2207,0.17395,561.0,17.0,0.000000,11000,15600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86913,k8691300900493934,49393,31/05/2023,13/02/2025,5000,40,Femenino,0,72770,72770.0,...,1,0.05814,0.0,P1594A,0.04904,657.0,16.0,1.760470,4000,8000.0
94056,k9405653593,53593,19/07/2023,15/12/2024,5000,29,Femenino,0,72840,72803.0,...,1,0.10624,0.0,P1594A,1.40224,661.0,3.0,1.515980,6000,6000.0
101852,k10185257870,57870,09/09/2023,28/11/2024,3000,19,Femenino,0,72803,72803.0,...,1,0.19025,0.0,P1594A,0.27397,657.0,5.0,1.147565,6500,6500.0


In [51]:
data.columns

Index(['referencia', 'id_cliente', 'Fecha Inicial', 'fecha_finalizada',
       'Monto', 'Edad Solicitud', 'Genero', 'Dependientes', 'CP', 'CP Aval',
       'Ingresos', 'Egresos', 'Score CC', 'Plazo', 'Categoria', 'Unidad',
       'Ciclo', 'id_credito_ant', 'Monto_Ant', 'tarifa_ant', 'pagado_cred_ant',
       'id_fecha_impresion_ant', 'id_fecha_finalizada_ant',
       'id_fecha_impresion_actual', 'id_fecha_finalizada_actual',
       'pagos_realizados_actual', 'vida_ant', 'Frecuencia_Ant', 'vida_actual',
       'Incobrable', 'score', 'Score_Ant', 'id_agente', 'score_agente',
       'sem_inicio', 'semanas', 'Score Agt', 'Deficit', 'IngresoPercapita'],
      dtype='object')

## 2.3 Agregar "Tiene aval" a partir de CP_Aval

In [52]:
if 'Tiene aval' in categorical_features:
    data['Tiene aval'] = data.apply(lambda x: 0 if pd.isnull(x['CP Aval']) else 1, axis=1)

## 2.3 Agregar Latitud y Longitud a partir del CP

In [53]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt,Deficit,IngresoPercapita
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.000000,-4000,0.0
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.000000,3000,6000.0
69758,k6975899900395917,39591,25/01/2023,12/05/2023,3000,18,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.17940,564.0,17.0,0.000000,3200,7200.0
72809,k7280999900413583,41358,15/02/2023,05/05/2023,3000,62,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.16130,567.0,17.0,0.170350,-3000,0.0
66301,k6630110001200377482,37748,02/01/2023,30/03/2023,3000,35,Femenino,0,94347,94347.0,...,0,0.20456,0.0,2207,0.17395,561.0,17.0,0.000000,11000,15600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86913,k8691300900493934,49393,31/05/2023,13/02/2025,5000,40,Femenino,0,72770,72770.0,...,1,0.05814,0.0,P1594A,0.04904,657.0,16.0,1.760470,4000,8000.0
94056,k9405653593,53593,19/07/2023,15/12/2024,5000,29,Femenino,0,72840,72803.0,...,1,0.10624,0.0,P1594A,1.40224,661.0,3.0,1.515980,6000,6000.0
101852,k10185257870,57870,09/09/2023,28/11/2024,3000,19,Femenino,0,72803,72803.0,...,1,0.19025,0.0,P1594A,0.27397,657.0,5.0,1.147565,6500,6500.0


In [54]:
if features_to_use.intersection(pkdp.latitud_longitud_features_set):
    
    if 'CP_Latitud'in features_to_use: 
        print("Seting CP_Latitud")
        data = cp_manager.set_geographic_coordinate(data,'CP', 'CP_Latitud', 'Latitud')
        print(data.columns)
        print("-----------------------")
    
    if 'CP_Longitud' in features_to_use: 
        print("Seting CP_Longitud")
        data = cp_manager.set_geographic_coordinate(data,'CP', 'CP_Longitud', 'Longitud')
        print(data.columns)
        print("-----------------------")

    if 'CP_Aval_Latitud' in features_to_use: 
        data = cp_manager.set_geographic_coordinate(data,'CP Aval','CP_Aval_Latitud', 'Latitud')
        print(data.columns)
        print("-----------------------")

    if 'CP_Aval_Longitud' in features_to_use: 
        data = cp_manager.set_geographic_coordinate(data,'CP Aval','CP_Aval_Longitud', 'Longitud')
        print(data.columns)
        print("-----------------------")

In [55]:
data

,referencia,id_cliente,Fecha Inicial,fecha_finalizada,Monto,Edad Solicitud,Genero,Dependientes,CP,CP Aval,...,Incobrable,score,Score_Ant,id_agente,score_agente,sem_inicio,semanas,Score Agt,Deficit,IngresoPercapita
id_solicitud,,,,,,,,,,,,,,,,,,,,,
71677,k7167799900407108,40710,08/02/2023,04/05/2023,8000,55,Femenino,0,94542,94542.0,...,0,0.20455,0.0,123879,0.00000,576.0,7.0,0.000000,-4000,0.0
121744,k12174468622,68622,31/01/2024,09/07/2024,3000,18,Femenino,0,90800,90800.0,...,1,0.63313,0.0,1309,0.57273,617.0,25.0,0.000000,3000,6000.0
69758,k6975899900395917,39591,25/01/2023,12/05/2023,3000,18,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.17940,564.0,17.0,0.000000,3200,7200.0
72809,k7280999900413583,41358,15/02/2023,05/05/2023,3000,62,Femenino,0,62748,62748.0,...,0,0.20456,0.0,2206,0.16130,567.0,17.0,0.170350,-3000,0.0
66301,k6630110001200377482,37748,02/01/2023,30/03/2023,3000,35,Femenino,0,94347,94347.0,...,0,0.20456,0.0,2207,0.17395,561.0,17.0,0.000000,11000,15600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86913,k8691300900493934,49393,31/05/2023,13/02/2025,5000,40,Femenino,0,72770,72770.0,...,1,0.05814,0.0,P1594A,0.04904,657.0,16.0,1.760470,4000,8000.0
94056,k9405653593,53593,19/07/2023,15/12/2024,5000,29,Femenino,0,72840,72803.0,...,1,0.10624,0.0,P1594A,1.40224,661.0,3.0,1.515980,6000,6000.0
101852,k10185257870,57870,09/09/2023,28/11/2024,3000,19,Femenino,0,72803,72803.0,...,1,0.19025,0.0,P1594A,0.27397,657.0,5.0,1.147565,6500,6500.0


In [56]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 26020 entries, 71677 to 124200
Data columns (total 39 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   referencia                  26020 non-null  object 
 1   id_cliente                  26020 non-null  int64  
 2   Fecha Inicial               26020 non-null  object 
 3   fecha_finalizada            18571 non-null  object 
 4   Monto                       26020 non-null  int32  
 5   Edad Solicitud              26020 non-null  int32  
 6   Genero                      26020 non-null  object 
 7   Dependientes                26020 non-null  int32  
 8   CP                          26020 non-null  int64  
 9   CP Aval                     25814 non-null  object 
 10  Ingresos                    26020 non-null  int32  
 11  Egresos                     26020 non-null  int32  
 12  Score CC                    26020 non-null  int32  
 13  Plazo                     

## 2.4 Agregar los odds de circulo de crédito

In [57]:
if ('Odds' in quantitative_features):
    score_bins = [-1, 235, 406, 433, 451, 465, 477, 489, 499, 509, 518, 528, 538, 548, 559, 571, 584, 598, 614, 634, 664, 861]
    odds = [0, 0.90, 1.20, 1.30, 1.50, 1.60, 1.80, 1.90, 2.00, 2.10, 2.20, 2.60, 2.75, 2.90, 3.20, 3.50, 4.00, 4.60, 5.30, 6.10, 8.30]

    data['Odds'] = pd.cut(data['Score CC'], score_bins, labels=odds)
    data['Odds'] = data['Odds'].astype(float)
    print(data[['Score CC', 'Odds']])

              Score CC  Odds
id_solicitud                
71677              438  1.30
121744               0  0.00
69758                0  0.00
72809              430  1.20
66301              412  1.20
...                ...   ...
86913              486  1.80
94056              603  4.60
101852             546  2.75
105378             490  1.90
124200             505  2.00

[26020 rows x 2 columns]


In [58]:
quantitative_features

{'Deficit',
 'Dependientes',
 'Edad Solicitud',
 'Egresos',
 'IngresoPercapita',
 'Ingresos',
 'Mes inicial',
 'Monto',
 'Odds',
 'Score CC'}

In [59]:
data.columns

Index(['referencia', 'id_cliente', 'Fecha Inicial', 'fecha_finalizada',
       'Monto', 'Edad Solicitud', 'Genero', 'Dependientes', 'CP', 'CP Aval',
       'Ingresos', 'Egresos', 'Score CC', 'Plazo', 'Categoria', 'Unidad',
       'Ciclo', 'id_credito_ant', 'Monto_Ant', 'tarifa_ant', 'pagado_cred_ant',
       'id_fecha_impresion_ant', 'id_fecha_finalizada_ant',
       'id_fecha_impresion_actual', 'id_fecha_finalizada_actual',
       'pagos_realizados_actual', 'vida_ant', 'Frecuencia_Ant', 'vida_actual',
       'Incobrable', 'score', 'Score_Ant', 'id_agente', 'score_agente',
       'sem_inicio', 'semanas', 'Score Agt', 'Deficit', 'IngresoPercapita',
       'Odds'],
      dtype='object')

## 2.5 Agregar el número de mes en que se solicita el crédito

In [60]:
if ('Mes inicial' in quantitative_features):
    data['Fecha Inicial'] = pd.to_datetime(data['Fecha Inicial'], format='%d/%m/%Y')
    data['Mes inicial'] = data['Fecha Inicial'].dt.month

## 2.6 Transformación de columnas categóricas representadas como números

### 2.4.1 Mapear los estudios del cliente de un id entero a su significado

In [61]:
if 'Estudios' in features_to_use:
    nivel_educativo = {
        0: 'Sin Estudios',
        1: 'Primaria',
        2: 'Secundaria',
        3: 'Preparatoria',
        4: 'Técnica',
        5: 'Licenciatura',
        6: 'Maestría',
        7: 'Carrera Trunca'
    }
    data['Estudios'] = data['Estudios'].map(nivel_educativo)

In [62]:
data['Plazo'].head(5)

id_solicitud
71677     17 Sem
121744    17 Sem
69758     17 Sem
72809     17 Sem
66301     17 Sem
Name: Plazo, dtype: object

In [63]:
if 'Plazo' in features_to_use:
    plazo_pago = {
        9: '9 Sem', 
        13: '13 Sem',
        16: '16 Sem',  
        17: '17 Sem',
        26: '26 Sem', 
        27: '27 Sem', 
    }
    data['Plazo'] = data['Plazo'].map(plazo_pago)

In [64]:
data['Plazo'].head(5)

id_solicitud
71677     17 Sem
121744    17 Sem
69758     17 Sem
72809     17 Sem
66301     17 Sem
Name: Plazo, dtype: object

## 2.7 Usar solamente un numero N de dígitos en el código postal

In [65]:
#if pkconfig.zip_digits <= 4:
#    if 'CP' in features_to_use: pk.transform_CP(data,'CP', pkconfig.zip_digits)
#    if 'CP Aval' in features_to_use: pk.transform_CP(data,'CP Aval', pkconfig.zip_digits)

## Borrar las columnas que no serán usados

In [66]:
data = data.drop(labels=features_to_drop, axis=1)
data.head()

,Fecha Inicial,Monto,Edad Solicitud,Genero,Dependientes,Ingresos,Egresos,Score CC,Incobrable,Deficit,IngresoPercapita,Odds,Mes inicial
id_solicitud,,,,,,,,,,,,,
71677,2023-02-08,8000,55,Femenino,0,0,4000,438,0,-4000,0.0,1.3,2
121744,2024-01-31,3000,18,Femenino,0,6000,3000,0,1,3000,6000.0,0.0,1
69758,2023-01-25,3000,18,Femenino,0,7200,4000,0,0,3200,7200.0,0.0,1
72809,2023-02-15,3000,62,Femenino,0,0,3000,430,0,-3000,0.0,1.2,2
66301,2023-01-02,3000,35,Femenino,0,15600,4600,412,0,11000,15600.0,1.2,1


In [67]:
pkdp.print_sizes(data)

Rows: 26020
Columns: 13
Max index: 264422


In [68]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 26020 entries, 71677 to 124200
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Fecha Inicial     26020 non-null  datetime64[ns]
 1   Monto             26020 non-null  int32         
 2   Edad Solicitud    26020 non-null  int32         
 3   Genero            26020 non-null  object        
 4   Dependientes      26020 non-null  int32         
 5   Ingresos          26020 non-null  int32         
 6   Egresos           26020 non-null  int32         
 7   Score CC          26020 non-null  int32         
 8   Incobrable        26020 non-null  int32         
 9   Deficit           26020 non-null  int32         
 10  IngresoPercapita  26020 non-null  float64       
 11  Odds              26020 non-null  float64       
 12  Mes inicial       26020 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int32(8), int64(1), object(1)
memory 

In [69]:
data.dtypes

Fecha Inicial       datetime64[ns]
Monto                        int32
Edad Solicitud               int32
Genero                      object
Dependientes                 int32
Ingresos                     int32
Egresos                      int32
Score CC                     int32
Incobrable                   int32
Deficit                      int32
IngresoPercapita           float64
Odds                       float64
Mes inicial                  int64
dtype: object

In [70]:
data.head()
#data.to_excel(pkconfig.data_file_name)
data.to_csv("./data/" + pkconfig.data_file_name)

In [71]:
data_tracker.to_dataframe().to_csv("./data/" + pkconfig.conf_column + "_data_tracking.csv")

In [72]:
features_to_drop


['Unidad',
 'fecha_finalizada',
 'semanas',
 'sem_inicio',
 'Score Agt',
 'Frecuencia_Ant',
 'id_fecha_finalizada_ant',
 'Monto_Ant',
 'vida_actual',
 'id_cliente',
 'referencia',
 'id_credito_ant',
 'score_agente',
 'Plazo',
 'Ciclo',
 'id_fecha_finalizada_actual',
 'CP Aval',
 'CP',
 'id_agente',
 'id_fecha_impresion_actual',
 'Score_Ant',
 'Categoria',
 'score',
 'tarifa_ant',
 'id_fecha_impresion_ant',
 'pagado_cred_ant',
 'vida_ant',
 'pagos_realizados_actual']

In [73]:
time_now = datetime.now()

format_date_hour = "%Y-%m-%d %H:%M:%S"
date_and_hour = time_now.strftime(format_date_hour)

# Imprimir la fecha y hora
print("Ended at:", date_and_hour)

Ended at: 2026-06-11 21:10:20


In [74]:
data_tracker.to_dataframe()

,Step,Rows,Columns,Column Names
0,Initialization,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
1,First filter for NV_CC_CS,113630,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
2,Second filter for NV_CC_CS,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
3,Cleaning datos conyuge,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
4,Cleaning egresos,48024,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
5,Cleaning for columns with null,48003,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
6,Cleaning int features with dropna,26091,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
7,Cleaning na floats,26091,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."
8,Cleaning numeric data min max,26020,37,"[referencia, id_cliente, Fecha Inicial, fecha_..."


In [75]:
data.head(60)

,Fecha Inicial,Monto,Edad Solicitud,Genero,Dependientes,Ingresos,Egresos,Score CC,Incobrable,Deficit,IngresoPercapita,Odds,Mes inicial
id_solicitud,,,,,,,,,,,,,
71677,2023-02-08,8000,55,Femenino,0,0,4000,438,0,-4000,0.000000,1.30,2
121744,2024-01-31,3000,18,Femenino,0,6000,3000,0,1,3000,6000.000000,0.00,1
69758,2023-01-25,3000,18,Femenino,0,7200,4000,0,0,3200,7200.000000,0.00,1
72809,2023-02-15,3000,62,Femenino,0,0,3000,430,0,-3000,0.000000,1.20,2
66301,2023-01-02,3000,35,Femenino,0,15600,4600,412,0,11000,15600.000000,1.20,1
67288,2023-01-09,3000,22,Femenino,1,18000,6500,482,0,11500,9000.000000,1.80,1
67327,2023-01-06,3000,22,Femenino,2,15500,6900,502,0,8600,5166.666667,2.00,1
67340,2023-01-06,5000,39,Masculino,2,25000,7500,389,0,17500,8333.333333,0.90,1
67355,2023-01-06,3000,30,Femenino,2,9500,5500,449,0,4000,3166.666667,1.30,1


In [76]:
data = data.reset_index(drop=True)

In [77]:
data.tail(60)

,Fecha Inicial,Monto,Edad Solicitud,Genero,Dependientes,Ingresos,Egresos,Score CC,Incobrable,Deficit,IngresoPercapita,Odds,Mes inicial
25960,2023-11-13,3000,33,Femenino,0,12000,0,684,1,12000,12000.0,8.30,11
25961,2023-12-22,3000,49,Femenino,0,10000,3500,447,1,6500,10000.0,1.30,12
25962,2024-01-03,3000,60,Femenino,0,10000,7000,561,1,3000,10000.0,3.20,1
25963,2024-01-08,5000,58,Femenino,0,10000,4000,518,1,6000,10000.0,2.10,1
25964,2024-01-22,3000,22,Femenino,0,10000,5000,352,1,5000,10000.0,0.90,1
25965,2024-01-30,5000,45,Femenino,0,13000,7000,436,1,6000,13000.0,1.30,1
25966,2024-01-30,3000,54,Femenino,0,12000,4500,523,1,7500,12000.0,2.20,1
25967,2024-02-05,3000,22,Femenino,0,6000,4000,413,1,2000,6000.0,1.20,2
25968,2024-02-02,3000,33,Femenino,0,12000,4000,427,1,8000,12000.0,1.20,2
25969,2024-02-14,3000,30,Femenino,0,8000,5000,631,1,3000,8000.0,5.30,2


In [78]:
# Imprimir la hora actual
print("Finalizado a las:", datetime.now().strftime("%H:%M:%S"))

# Hacer un sonido
if sys.platform.startswith("win"):
    import winsound
    winsound.Beep(1000, 500)  # Frecuencia de 1000 Hz, duración de 500 ms
    winsound.Beep(500, 500) 
    winsound.Beep(1000, 500) 
else:
    print('\a')  # Hace un beep en algunos sistemas UNIX
    
print('Configuration : ', pkconfig.conf_column)
pkdp.print_sizes(data)

Finalizado a las: 21:10:20
Configuration :  NV_CC_CS
Rows: 26020
Columns: 13
Max index: 26019
